# Prepare the LAION-Art AMP attack set

Assign each clean image its highest-scoring caption concept using AMP's concept-selection procedure, then reproducibly sample source-target pairs. This notebook only prepares clean inputs; it does not generate adversarial images.

In [1]:
from pathlib import Path
import json
import random
import shutil

import nltk
import numpy as np
import open_clip
import pandas as pd
import spacy
import torch
from PIL import Image
from nltk.stem import WordNetLemmatizer
from tqdm.auto import tqdm

DATA_DIR = Path("dataset/laion_art")
CLEAN_DIR = DATA_DIR / "clean"
CAPTIONS_PATH = DATA_DIR / "llava_captions.csv"
ASSIGNMENTS_PATH = DATA_DIR / "concept_assignments.csv"
PAIRS_PATH = DATA_DIR / "concept_pairs.csv"
ATTACK_DIR = DATA_DIR / "attack_set"

NUM_CONCEPT_PAIRS = 25
IMAGES_PER_PAIR = 16
RANDOM_SEED = 42
IMAGE_BATCH_SIZE = 32
TEXT_BATCH_SIZE = 256
REBUILD_ASSIGNMENTS = False
MIN_SELECTED_CONCEPT_SCORE = 0.99

MODEL_NAME = "EVA02-E-14-plus"
PRETRAINED = "laion2b_s9b_b144k"
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("A CUDA GPU is required for efficient concept scoring.")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
print(f"Using {device}: {torch.cuda.get_device_name(0)}")

Using cuda: NVIDIA GeForce RTX 5090


In [2]:
# Load LLaVA captions and match them to clean image stems.
captions = pd.read_csv(CAPTIONS_PATH, dtype={"image_id": str})
images = sorted(CLEAN_DIR.glob("*.png"))
if not images:
    raise FileNotFoundError(f"No PNG images found in {CLEAN_DIR}")

required_columns = {"image_id", "llava_caption"}
missing_columns = required_columns - set(captions.columns)
if missing_columns:
    raise ValueError(f"Missing columns in {CAPTIONS_PATH}: {sorted(missing_columns)}")
if captions["image_id"].duplicated().any():
    raise ValueError(f"Duplicate image IDs in {CAPTIONS_PATH}")
caption_by_id = captions.set_index("image_id")["llava_caption"]
records = pd.DataFrame({
    "image_path": [path.as_posix() for path in images],
    "image_id": [path.stem for path in images],
})
records["caption"] = records["image_id"].map(caption_by_id)
missing_caption = records["caption"].isna() | records["caption"].fillna("").str.strip().eq("")
if missing_caption.any():
    missing = records.loc[missing_caption, "image_id"].tolist()[:10]
    raise ValueError(f"Missing LLaVA captions for image IDs: {missing}")

rebuild_assignments = REBUILD_ASSIGNMENTS or not ASSIGNMENTS_PATH.exists()
if not rebuild_assignments:
    cached = pd.read_csv(ASSIGNMENTS_PATH, dtype={"image_id": str})
    cached_captions = cached.set_index("image_id")["caption"]
    current_captions = records.set_index("image_id")["caption"]
    rebuild_assignments = not cached_captions.equals(current_captions)
    if rebuild_assignments:
        print("Caption source changed; rebuilding concept assignments.")
records.head()

,image_path,image_id,caption
0,dataset/laion_art/clean/000000000000.png,000000000000,A red bridge over a river.
1,dataset/laion_art/clean/000000000001.png,000000000001,A young boy standing in front of a wooden stru...
2,dataset/laion_art/clean/000000000004.png,000000000004,A plate of food with a bowl of salsa.
3,dataset/laion_art/clean/000000000006.png,000000000006,A pink vintage car with a wooden panel on the ...
4,dataset/laion_art/clean/000000000007.png,000000000007,A snowy scene with a large brown bear holding ...


In [3]:
# AMP extracts NOUN/PROPN tokens, lowercases them, and applies WordNet lemmatization.
nltk.download("wordnet", quiet=True)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
lemmatizer = WordNetLemmatizer()

def extract_concepts(caption):
    nouns = {token.text for token in nlp(str(caption)) if token.pos_ in {"NOUN", "PROPN"}}
    return sorted({lemmatizer.lemmatize(noun.lower()) for noun in nouns})

if rebuild_assignments:
    records["candidate_concepts_list"] = [
        extract_concepts(caption) for caption in tqdm(records["caption"], desc="Extracting concepts")
    ]
    records["assignable"] = records["candidate_concepts_list"].map(bool)
else:
    print(f"Using existing {ASSIGNMENTS_PATH}; set REBUILD_ASSIGNMENTS=True to recompute.")

/home/anantaraha/amp/.venv/lib/python3.10/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/anantaraha/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


Extracting concepts:   0%|          | 0/59277 [00:00<?, ?it/s]

In [4]:
# Encode each distinct concept once, then score image batches on the GPU.
if rebuild_assignments:
    model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME,
        pretrained=PRETRAINED,
        precision="fp16",
        device=device,
    )
    model.eval()
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)

    assignable_records = records.loc[records["assignable"]]
    vocabulary = sorted({concept for concepts in assignable_records["candidate_concepts_list"] for concept in concepts})
    text_feature_batches = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(vocabulary), TEXT_BATCH_SIZE), desc="Encoding concepts"):
            tokens = tokenizer(vocabulary[start:start + TEXT_BATCH_SIZE]).to(device)
            features = model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)
            text_feature_batches.append(features)
    text_features = torch.cat(text_feature_batches) if text_feature_batches else None
    concept_index = {concept: index for index, concept in enumerate(vocabulary)}

    selected, scores = [], []
    with torch.inference_mode():
        for start in tqdm(range(0, len(assignable_records), IMAGE_BATCH_SIZE), desc="Scoring images"):
            batch = assignable_records.iloc[start:start + IMAGE_BATCH_SIZE]
            pixels = torch.stack([
                preprocess(Image.open(path).convert("RGB")) for path in batch["image_path"]
            ]).to(device=device, dtype=torch.float16)
            image_features = model.encode_image(pixels)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            similarities = 100.0 * image_features @ text_features.T

            for row_index, concepts in enumerate(batch["candidate_concepts_list"]):
                indices = torch.tensor([concept_index[concept] for concept in concepts], device=device)
                probabilities = similarities[row_index, indices].softmax(dim=-1)
                best = int(probabilities.argmax())
                selected.append(concepts[best])
                scores.append(float(probabilities[best].cpu()))

    records["candidate_concepts"] = records["candidate_concepts_list"].map(json.dumps)
    records["selected_concept"] = pd.Series(selected, index=assignable_records.index)
    records["score"] = pd.Series(scores, index=assignable_records.index)
    records[["image_path", "image_id", "caption", "candidate_concepts", "assignable", "selected_concept", "score"]].to_csv(
        ASSIGNMENTS_PATH, index=False
    )

assignments = pd.read_csv(ASSIGNMENTS_PATH, dtype={"image_id": str})
assignable_count = assignments["selected_concept"].notna().sum()
print(f"Assignable images: {assignable_count}; skipped: {len(assignments) - assignable_count}")
assignments.head()

Encoding concepts:   0%|          | 0/9 [00:00<?, ?it/s]

Scoring images:   0%|          | 0/1853 [00:00<?, ?it/s]

Assignable images: 59276; skipped: 1


,image_path,image_id,caption,candidate_concepts,assignable,selected_concept,score
0,dataset/laion_art/clean/000000000000.png,000000000000,A red bridge over a river.,"[""bridge"", ""river""]",True,bridge,0.979004
1,dataset/laion_art/clean/000000000001.png,000000000001,A young boy standing in front of a wooden stru...,"[""boy"", ""front"", ""hay"", ""pile"", ""structure""]",True,hay,0.887207
2,dataset/laion_art/clean/000000000004.png,000000000004,A plate of food with a bowl of salsa.,"[""bowl"", ""food"", ""plate"", ""salsa""]",True,food,0.768555
3,dataset/laion_art/clean/000000000006.png,000000000006,A pink vintage car with a wooden panel on the ...,"[""car"", ""panel"", ""side""]",True,car,0.878418
4,dataset/laion_art/clean/000000000007.png,000000000007,A snowy scene with a large brown bear holding ...,"[""bear"", ""gift"", ""scene""]",True,gift,0.880859


In [5]:
# Compute the top 100, then apply AMP's confidence threshold for sampling.
concept_counts = assignments["selected_concept"].value_counts()
top_100 = concept_counts.head(100)
qualified_assignments = assignments[assignments["score"].gt(MIN_SELECTED_CONCEPT_SCORE)]
qualified_counts = qualified_assignments["selected_concept"].value_counts()
eligible = qualified_counts.reindex(top_100.index, fill_value=0)
eligible = eligible[eligible >= IMAGES_PER_PAIR]
required_concepts = 2 * NUM_CONCEPT_PAIRS
if len(eligible) < required_concepts:
    raise ValueError(
        f"Need {required_concepts} top-100 concepts with at least {IMAGES_PER_PAIR} images above "
        f"score {MIN_SELECTED_CONCEPT_SCORE}; found {len(eligible)}."
    )

top_100.rename("frequency").rename_axis("concept").reset_index().head(100)

,concept,frequency
0,cake,3666
1,car,3093
2,doll,2298
3,woman,1678
4,dress,1260
...,...,...
95,sprinkle,121
96,cabinet,119
97,bouquet,118
98,sweater,117


In [6]:
# Seeded sampling supplies the image-level pairing rule not specified by AMP.
rng = np.random.default_rng(RANDOM_SEED)
chosen_concepts = rng.choice(eligible.index.to_numpy(), size=required_concepts, replace=False)
concept_pairs = pd.DataFrame({
    "pair_id": [f"pair_{index:02d}" for index in range(NUM_CONCEPT_PAIRS)],
    "source_concept": chosen_concepts[0::2],
    "target_concept": chosen_concepts[1::2],
})
concept_pairs["source_frequency"] = concept_pairs["source_concept"].map(qualified_counts)
concept_pairs["target_frequency"] = concept_pairs["target_concept"].map(qualified_counts)
concept_pairs.to_csv(PAIRS_PATH, index=False)
concept_pairs

,pair_id,source_concept,target_concept,source_frequency,target_frequency
0,pair_00,statue,juice,25,75
1,pair_01,beach,woman,120,272
2,pair_02,pumpkin,sprinkle,50,42
3,pair_03,tent,bike,221,37
4,pair_04,bread,cabinet,93,66
5,pair_05,drink,men,28,26
6,pair_06,tea,food,46,159
7,pair_07,santa,pancake,30,258
8,pair_08,table,bus,20,108
9,pair_09,fruit,window,220,22


In [7]:
# Rebuild the attack-set directories so the manifest and copied files stay in sync.
source_dir = ATTACK_DIR / "source"
target_dir = ATTACK_DIR / "target"
for directory in (source_dir, target_dir):
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)

manifest_rows = []
for pair in concept_pairs.itertuples(index=False):
    source_pool = qualified_assignments[qualified_assignments["selected_concept"].eq(pair.source_concept)]
    target_pool = qualified_assignments[qualified_assignments["selected_concept"].eq(pair.target_concept)]
    source_seed = int(rng.integers(0, 2**32 - 1))
    target_seed = int(rng.integers(0, 2**32 - 1))
    source_rows = source_pool.sample(IMAGES_PER_PAIR, random_state=source_seed).reset_index(drop=True)
    target_rows = target_pool.sample(IMAGES_PER_PAIR, random_state=target_seed).reset_index(drop=True)

    for image_index in range(IMAGES_PER_PAIR):
        sample_id = f"{pair.pair_id}_{image_index:02d}"
        source = source_rows.iloc[image_index]
        target = target_rows.iloc[image_index]
        source_output = source_dir / f"{sample_id}.png"
        target_output = target_dir / f"{sample_id}.png"
        shutil.copy2(source["image_path"], source_output)
        shutil.copy2(target["image_path"], target_output)
        manifest_rows.append({
            "sample_id": sample_id,
            "pair_id": pair.pair_id,
            "source_path": source_output.as_posix(),
            "target_path": target_output.as_posix(),
            "source_image_id": source["image_id"],
            "target_image_id": target["image_id"],
            "source_score": source["score"],
            "target_score": target["score"],
            "source_concept": pair.source_concept,
            "target_concept": pair.target_concept,
            "source_caption": source["caption"],
            "target_caption": target["caption"],
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(ATTACK_DIR / "manifest.csv", index=False)
manifest

,sample_id,pair_id,source_path,target_path,source_image_id,target_image_id,source_score,target_score,source_concept,target_concept,source_caption,target_caption
0,pair_00_00,pair_00,dataset/laion_art/attack_set/source/pair_00_00...,dataset/laion_art/attack_set/target/pair_00_00...,000000092688,000000089988,0.990234,0.999512,statue,juice,A statue of a man holding a surfboard.,Four bottles of juice and a lemon.
1,pair_00_01,pair_00,dataset/laion_art/attack_set/source/pair_00_01...,dataset/laion_art/attack_set/target/pair_00_01...,000000021922,000000051138,0.997070,0.995605,statue,juice,A statue of a woman wearing a crown.,A variety of colorful juices in glasses.
2,pair_00_02,pair_00,dataset/laion_art/attack_set/source/pair_00_02...,dataset/laion_art/attack_set/target/pair_00_02...,000000078682,000000062452,0.999023,0.998535,statue,juice,Two wooden statues of men kissing.,A glass of orange juice is next to a glass of ...
3,pair_00_03,pair_00,dataset/laion_art/attack_set/source/pair_00_03...,dataset/laion_art/attack_set/target/pair_00_03...,000000061596,000000084201,1.000000,0.998047,statue,juice,Two religious statues are standing next to eac...,Two glasses of red juice on a table.
4,pair_00_04,pair_00,dataset/laion_art/attack_set/source/pair_00_04...,dataset/laion_art/attack_set/target/pair_00_04...,000000025893,000000072171,1.000000,0.997559,statue,juice,A statue of a man holding a blue stick.,Three bottles of juice and a sliced apple and ...
...,...,...,...,...,...,...,...,...,...,...,...,...
395,pair_24_11,pair_24,dataset/laion_art/attack_set/source/pair_24_11...,dataset/laion_art/attack_set/target/pair_24_11...,000000041745,000000016008,0.994629,0.997559,variety,bird,A variety of cookies and candies on a table.,A red bird is perched on a black and red vase.
396,pair_24_12,pair_24,dataset/laion_art/attack_set/source/pair_24_12...,dataset/laion_art/attack_set/target/pair_24_12...,000000020037,000000029395,0.999512,0.998535,variety,bird,A table with a variety of cookies and a spoon.,A red bird perched on a branch.
397,pair_24_13,pair_24,dataset/laion_art/attack_set/source/pair_24_13...,dataset/laion_art/attack_set/target/pair_24_13...,000000015706,000000026085,0.997559,0.993652,variety,bird,A red plate with a variety of cookies and cand...,A red and black bird perched on a branch.
398,pair_24_14,pair_24,dataset/laion_art/attack_set/source/pair_24_14...,dataset/laion_art/attack_set/target/pair_24_14...,000000012079,000000018781,0.991699,0.998047,variety,bird,"A room with a table and chairs, decorated with...",Three crocheted birds in cups.
